## App Scheduler – Stop / Start SN UX Table Editor

This notebook is used by a **scheduled Databricks job** to stop or start the `sn-ux-db-table-editor` app outside business hours to save costs.

| Parameter | Description |
| --- | --- |
| `action` | `stop` or `start` |
| `app_name` | Name of the Databricks App (default: `sn-ux-db-table-editor`) |

In [0]:
dbutils.widgets.dropdown("action", "stop", ["stop", "start"], "Action")
dbutils.widgets.text("app_name", "sn-ux-db-table-editor", "App Name")

In [0]:
import requests
import time
import json

action = dbutils.widgets.get("action")
app_name = dbutils.widgets.get("app_name")

# ── Auth: use notebook context token ──
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
host = ctx.apiUrl().get()
token = ctx.apiToken().get()
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

print(f"Action:   {action}")
print(f"App name: {app_name}")
print(f"Host:     {host}")
print("-" * 50)

# ── Get current app status ──
# API returns two status objects:
#   compute_status.state : STOPPED → STARTING → ACTIVE  (compute lifecycle)
#   app_status.state     : UNAVAILABLE → RUNNING         (app health)
# For STOP  → wait for compute_status == STOPPED
# For START → wait for app_status == RUNNING  (means app is fully serving)
def get_app_status():
    resp = requests.get(f"{host}/api/2.0/apps/{app_name}", headers=headers)
    resp.raise_for_status()
    data = resp.json()
    compute_state = data.get("compute_status", {}).get("state", "UNKNOWN")
    compute_msg   = data.get("compute_status", {}).get("message", "")
    app_state     = data.get("app_status", {}).get("state", "UNKNOWN")
    return compute_state, compute_msg, app_state

current_compute, current_msg, current_app = get_app_status()
print(f"Compute state: {current_compute}")
print(f"App state:     {current_app}")
print(f"Message:       {current_msg}")

# ── Stop ──
if action == "stop":
    if current_compute in ("ACTIVE", "STARTING"):
        print(f"\nStopping '{app_name}'...")
        resp = requests.post(f"{host}/api/2.0/apps/{app_name}/stop", headers=headers)
        resp.raise_for_status()
        print(f"Stop request sent. Waiting for compute STOPPED...")
        for i in range(30):
            time.sleep(10)
            comp, msg, app_st = get_app_status()
            print(f"  [{i+1}] Compute: {comp} | App: {app_st}")
            if comp == "STOPPED":
                print(f"\n\u2705 App '{app_name}' stopped successfully.")
                break
        else:
            print(f"\n\u26a0\ufe0f App did not reach STOPPED state within 5 minutes.")
    elif current_compute == "STOPPED":
        print(f"\nApp is already stopped. Nothing to do.")
    else:
        print(f"\nApp compute is in state '{current_compute}'. Nothing to stop.")

# ── Start ──
elif action == "start":
    if current_compute == "STOPPED":
        print(f"\nStarting '{app_name}'...")
        resp = requests.post(f"{host}/api/2.0/apps/{app_name}/start", headers=headers)
        resp.raise_for_status()
        print(f"Start request sent. Waiting for app RUNNING...")
        for i in range(30):
            time.sleep(10)
            comp, msg, app_st = get_app_status()
            print(f"  [{i+1}] Compute: {comp} | App: {app_st}")
            if app_st == "RUNNING":
                print(f"\n\u2705 App '{app_name}' started and fully running.")
                break
        else:
            # Check if compute at least came up
            comp, _, app_st = get_app_status()
            if comp == "ACTIVE":
                print(f"\n\u26a0\ufe0f Compute is ACTIVE but app not yet RUNNING. It may still be deploying.")
            else:
                print(f"\n\u26a0\ufe0f App did not reach RUNNING state within 5 minutes.")
    elif current_compute == "ACTIVE" and current_app == "RUNNING":
        print(f"\nApp is already running. Nothing to do.")
    else:
        print(f"\nApp compute is in state '{current_compute}' / app '{current_app}'. Cannot start.")

else:
    raise ValueError(f"Invalid action '{action}'. Must be 'stop' or 'start'.")

In [0]:
final_state, final_msg, final_app = get_app_status()
print(f"Final compute status: {final_state}")
print(f"Final app status:     {final_app}")
print(f"Message:              {final_msg}")